In [ ]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI
import json
import pandas as pd

from sklearn.model_selection import train_test_split

label_map = {
    "Low viral": 0,
    "Medium viral": 1,
    "High viral": 2
}

# reverse map: 0->Low viral, 1->Medium viral, 2->High viral
reverse_label_map = {v: k for k, v in label_map.items()}


DEEPSEEK_API_KEY='...'
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")


def get_response(msg_text,system_prompt,max_tokens=20):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": msg_text},
        ],
        temperature=0,          # more consistent scoring
        max_tokens=max_tokens,
        stream=False,
        response_format={
            'type': 'json_object'
        }
    )
    

    raw = response.choices[0].message.content.strip()
    return raw,response

    # Optional: validate JSON output


    


In [ ]:
trial_num=29
trials=30
for trial_num in range(trial_num,trials):
    #print(build_query_string)
    print(f'trial:{trial_num}')
    with open(f'Out/trial{trial_num}/sys_prompt.txt','r') as f:
        system_prompt=f.read()
    
    df=pd.read_csv('train_df_test.csv')
    res=[]
    for idx,row in df.iterrows():
        #print(row['Text'])
        msg_text=row['Text']
        raw,_=get_response(msg_text,system_prompt)
        result = json.loads(raw)
        #print(raw,result)        
        res.append({
            'ID':row['ID'],
            'label':int(result['virality_score'])
        })
        res_df=pd.DataFrame(res)
        res_df.to_csv(f'Out/trial{trial_num}/deepseek-chat_test_Text_labels.csv',index=False)    
        



### For the test dataset 16/02/2026

- Checked from Eval that best performance is for trial 7
- Use the same system prompt

In [37]:
trial_num=0
with open(f'Out/trial{trial_num}/sys_prompt.txt','r') as f:
    system_prompt=f.read()
print(system_prompt)    

test_loc='/Users/ashhadulislam/projects/other_misc/NakbaVirality/nakba_virality_test/'
df=pd.read_csv(f'{test_loc}/TestSet.csv')
res=[]
for idx,row in df.iterrows():
    
    #print(row['Text'])
    msg_text=row['Text']
    raw,_=get_response(msg_text,system_prompt)
    result = json.loads(raw)
    #print(raw,result)        
    res.append({
        'ID':row['ID'],
        'label':int(result['virality_score'])        
    })
    if idx%50==0:
        print(f'{idx}/{df.shape[0]}')
        res_df=pd.DataFrame(res)
        res_df.to_csv(f'Out/trial{trial_num}/deepseek-chat_TestSet.csv',index=False)    





You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.

0/872
50/872
100/872
150/872
200/872
250/872
300/872
350/872
400/872
450/872
500/872
550/872
600/872
650/872
700/872
750/872
800/872
850/872


In [38]:
res_df=pd.DataFrame(res)
res_df.to_csv(f'Out/trial{trial_num}/deepseek-chat_TestSet.csv',index=False)    

In [39]:

file_preds=f'Out/trial{trial_num}/deepseek-chat_TestSet.csv'
df_preds = pd.read_csv(file_preds)
if 'ViralityLabel' in df_preds.columns:
    df_preds["label"]=df_preds["ViralityLabel"]
    df_preds=df_preds.drop(columns=['ViralityLabel'])


# replace numeric labels with strings
df_preds["label"] = pd.to_numeric(df_preds["label"], errors="coerce").map(reverse_label_map)

df_preds.head()
df_preds.to_csv(f'Out/trial{trial_num}/predictions.csv',index=False)    

In [ ]:
df_preds.shape

## Test the openrouter now

In [ ]:
import requests
import json




import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from collections import Counter

label_map = {
    "Low viral": 0,
    "Medium viral": 1,
    "High viral": 2
}

import os
import json
import pandas as pd

import numpy as np
# reverse map: 0->Low viral, 1->Medium viral, 2->High viral
reverse_label_map = {v: k for k, v in label_map.items()}

openrouter_api_key='...'
def openrouter_gpt52_get_response(msg_text, system_prompt, max_tokens=200):
    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {openrouter_api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": "openai/gpt-5.2",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": msg_text},
                ],
                "response_format": {"type": "json_object"},
                "temperature": 0,
                "max_tokens": max_tokens,
            },
            timeout=30,
        )

        response.raise_for_status()

        result = response.json()
        content = result["choices"][0]["message"]["content"]
        parsed = json.loads(content)

        return parsed

    except Exception as e:
        # optional: log the error
        # print(f"OpenRouter request failed: {e}")
        return None

### For the test dataset 16/02/2026

- Checked from Eval that deepseek best performance is for trial 7
- Use the same system prompt from deepseek and use it on openai gpt 5.2

In [35]:
model_name='openai/gpt-5.2'
trial_num=0
with open(f'Out/trial{trial_num}/sys_prompt.txt','r') as f:
    system_prompt=f.read()
print(system_prompt)    

test_loc='/Users/ashhadulislam/projects/other_misc/NakbaVirality/nakba_virality_test/'
df=pd.read_csv(f'{test_loc}/TestSet.csv')
res=[]
for idx,row in df.iterrows():
    
    #print(row['Text'])
    msg_text=row['Text']
    resp=openrouter_gpt52_get_response(msg_text,system_prompt)
    resp=resp["virality_score"]
    res.append({
        'ID':row['ID'],
        'label':int(resp)     
    })

    if idx%50==0:
        print(f'{idx}/{df.shape[0]}')
        res_df=pd.DataFrame(res)
        res_df["label"] = pd.to_numeric(res_df["label"], errors="coerce").map(reverse_label_map)        
        res_df.to_csv(f'Out/{model_name}/trial{trial_num}_TestSet.csv',index=False)
        



res_df=pd.DataFrame(res)
res_df["label"] = pd.to_numeric(res_df["label"], errors="coerce").map(reverse_label_map)        
res_df.to_csv(f'Out/{model_name}/trial{trial_num}_TestSet.csv',index=False)



You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.

0/872
50/872
100/872
150/872
200/872
250/872
300/872
350/872
400/872
450/872
500/872
550/872
600/872
650/872
700/872
750/872
800/872
850/872


In [34]:
f'Out/{model_name}/trial{trial_num}_TestSet.csv'

'Out/openai/gpt-5.2/trial7_TestSet.csv'

In [ ]:
model_name='openai/gpt-5.2'

trial_num=7

df_test=pd.read_csv('train_df_test.csv')

for trial_num in range(0,trials):
    #print(build_query_string)
    print(f'trial:{trial_num}')
    if not os.path.isfile(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt'):
        continue
    with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','r') as f:
        system_prompt=f.read()
    #print(system_prompt)
    res=[]
    for index,row in df_test.iterrows():
        ID=row['ID']
        text=row['Text']
        resp=openrouter_gpt52_get_response(text,system_prompt)
        if not resp:
            continue
        resp=resp["virality_score"]
        res.append({
            'ID':row['ID'],
            'label':int(resp)     
        })
        res_df=pd.DataFrame(res)
        res_df.to_csv(f'Out/{model_name}/trial{trial_num}/openrouter_openai_test.csv',index=False)
              
    
    # improved_system_prompt_raw=openrouter_gpt52_get_response(build_query_string,system_prompt,max_tokens=500)
    # improved_system_prompt_raw=improved_system_prompt_raw['improved_system_prompt']
    # improved_system_prompt_raw=openrouter_gpt52_get_response(summarise_prompt,improved_system_prompt_raw,max_tokens=400)
    # improved_system_prompt = improved_system_prompt_raw['improved_system_prompt']

    # os.makedirs(f'Out/{model_name}/trial{trial_num}/',exist_ok=True)
    # with open(f'Out/{model_name}/trial{trial_num}/sys_prompt.txt','w') as f:
    #     f.write(improved_system_prompt)


In [ ]:
index

In [ ]:
text

In [ ]:
openrouter_gpt52_get_response(text,system_prompt)